# Model characteristics
- Module validity verification
- Convergence investigation
- Robustness investigation

In [ ]:
import torch
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from typing import Dict, Optional, Tuple, Union

from utils.metrics import PIVmetric
from utils.visualizer import plot_flow_field
from matplotlib.ticker import MultipleLocator

# 数据加载：事件、光流
from generator.flow import FlowManager
from generator.images import config_events
from generator.loader import EventManager


from utils.add_noise import add_event_noise
from utils.painter import extract_data, plot_curvegraph, plot_loss_curves, analyze_error, plot_exp111

# from methods import compute_flow_quantities
from gaussian_flow import GaussianFlowConfig, run_gaussian_flow

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)              
torch.cuda.manual_seed(seed)         
torch.cuda.manual_seed_all(seed)   

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
flow0 = FlowManager(mode="data", data_path="./data/datasets/JHTDB_isotropic1024_hd_00001_flow.npz")
flow1 = FlowManager(mode="data", data_path="./data/datasets/DNS_turbulence_00001_flow.npz")
flow2 = FlowManager(mode="data", data_path="./data/datasets/SQG_00001_flow.npz")

# flow0 = FlowManager(mode="analytical", name="cellular_flow", vmax=10, p=256)
# flow1 = FlowManager(mode="analytical", name="lamb_oseen", x_c=128, y_c=128, Gamma=5e3, rc=60)
# flow2 = FlowManager(mode="analytical", name="shear_flow", vmax=10, p=256)

flow_funcs = [flow0, flow1, flow2]
flow_names = ["JHTDB-isotropic", "DNS-turbulence", "SQG"]

## Module validity verification

In [ ]:
exp111_results = {}

for flow_index in tqdm(range(3), desc="Flows", position=0):

    flow_name = f"flow{flow_index}"
    exp111_results[flow_name] = {}

    # ================= 数据生成 =================
    flow_func = globals()[flow_name].evaluate

    cfg_evts = config_events()
    cfg_evts.time_length = 5
    cfg_evts.d = 0.5
    cfg_evts.density = 0.05
    
    event_loader = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func)

    evts, (u, v) = event_loader.load()
    flowgt = torch.stack([u, v], dim=0).to(device)
    
    events = evts[(evts[:, 2] > 0) & (evts[:, 2] < 3)]
    h_gt = u.cpu().numpy()
    w_gt = v.cpu().numpy()

    configs = [
        (enable_jacobian, init_n)
        for enable_jacobian in [1, 0]
        for init_n in range(50, 510, 50)
    ]

    for enable_jacobian, init_n in tqdm(configs, desc=f"{flow_name}", leave=False, position=1):

        jac_key = str(enable_jacobian)
        if jac_key not in exp111_results[flow_name]:
            exp111_results[flow_name][jac_key] = {}

        rmse_list = []
        aee_list = []

        # ===== repeat 5 次 =====
        for _ in tqdm(range(10), desc="repeat", leave=False, position=2):

            cfg = GaussianFlowConfig(verbose=False, init_n=init_n, enable_jacobian=enable_jacobian, phase=[500, 200, 250], iterations=300, lr_j=0.001)

            res = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)

            pred_h, pred_w = res.flow[0], res.flow[1]
            rmse, aee = PIVmetric(pred_h, pred_w, h_gt, w_gt, show=False)[:2]

            rmse_list.append(rmse)
            aee_list.append(aee)

        rmse_mean = float(np.mean(sorted(rmse_list)[1:-1]))
        aee_mean = float(np.mean(sorted(aee_list)[1:-1]))

        exp111_results[flow_name][jac_key][str(init_n)] = {"rmse": rmse_mean, "aee": aee_mean} 

In [ ]:
plot_exp111(exp111_results, metric="rmse")
plot_exp111(exp111_results, metric="aee")

## Convergence investigation

In [ ]:
gt_list=[]
flow_list=[]

for i in range(3):

    flow_func = globals()[f"flow{i}"].evaluate

    cfg_evts = config_events()
    cfg_evts.time_length = 5
    cfg_evts.d = 0.5
    cfg_evts.density = 0.05
    
    event_loader = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func)

    evts, (u, v) = event_loader.load()
    flowgt = torch.stack([u, v], dim=0).cpu()
    gt_list.append(flowgt)
    
    events = evts[(evts[:, 2] > 0) & (evts[:, 2] < 3)]
    h_gt = u.cpu().numpy()
    w_gt = v.cpu().numpy()

    cfg = GaussianFlowConfig(iterations=50, lr_j=0.05)

    res = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
    pred_h, pred_w = res.flow[0], res.flow[1]
    
    flow_pred = torch.tensor(np.stack([pred_h, pred_w], axis=0))
    flow_list.append(flow_pred)
    
    flow_stack = torch.stack(flow_list, axis=0)
    gt_stack = torch.stack(gt_list, axis=0)

analyze_error(flow_stack, gt_stack, flow_names, n_bins=10)

In [ ]:
history = []

for flow_index in range(3):

    flow_n = f"flow{flow_index}"
    flow_func = globals()[flow_n].evaluate

    cfg_evts = config_events()
    cfg_evts.time_length = 5
    cfg_evts.d = 0.5
    cfg_evts.density = 0.05
    
    event_loader = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func)
    evts, (u, v) = event_loader.load()

    events = evts[(evts[:, 2] > 0) & (evts[:, 2] < 3)]
    h_gt = u.cpu().numpy()
    w_gt = v.cpu().numpy()

    cfg = GaussianFlowConfig()

    res = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
    history.append(res.history)

plot_loss_curves(history, flow_names)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import trim_mean   # 用于修剪平均

def analyze_error_robust(pred_flows_list, gt_flows_list, flow_names, n_bins=5, trim=0.1):
    """
    多轮实验的鲁棒误差分析。
    pred_flows_list: list of length num_trials, 每个元素是 [C,2,H,W] 的预测光流张量
    gt_flows_list:   list of length num_trials, 每个元素是 [C,2,H,W] 的真值光流张量
    flow_names: 每个通道的名称列表，长度 C
    n_bins: 速度百分比分箱数
    trim: 去掉最高和最低的比例 (0.1 表示去掉各 10%)
    """
    num_trials = len(pred_flows_list)
    C = pred_flows_list[0].shape[0]

    # 首先收集所有 trial 中每个通道、每个 bin 的 EPE 值（标量）
    # 结构: epe_by_trial[C][bin_idx] = list of EPE values over trials
    epe_by_trial = [[[] for _ in range(n_bins)] for _ in range(C)]

    for trial_idx in range(num_trials):
        pred_flow = pred_flows_list[trial_idx]   # [C,2,H,W]
        gt_flow   = gt_flows_list[trial_idx]

        C, _, H, W = pred_flow.shape
        gt_mag_all = gt_flow.norm(dim=1)  # [C, H, W]

        # 所有通道共用同一个速度区间边界（基于所有 trial 的所有像素确定，避免边界漂移）
        if trial_idx == 0:
            v_min = gt_mag_all.min().item()
            v_max = gt_mag_all.max().item()
            bins = torch.linspace(v_min, v_max, n_bins + 1)
        else:
            # 使用第一次确定的 bins，保证不同 trial 的区间一致
            pass

        for c in range(C):
            gt_mag = gt_mag_all[c]
            pred = pred_flow[c]
            epe_map = (pred - gt_flow[c]).norm(dim=0)  # [H, W]

            for i in range(n_bins):
                lo = bins[i].item()
                hi = bins[i+1].item()
                mask = (gt_mag >= lo) & (gt_mag < hi)
                if mask.sum() == 0:
                    avg_epe = 0.0
                else:
                    avg_epe = epe_map[mask].mean().item()
                epe_by_trial[c][i].append(avg_epe)

    # 对每个通道、每个 bin 进行修剪平均
    final_avg_epe = np.zeros((C, n_bins))
    for c in range(C):
        for i in range(n_bins):
            values = epe_by_trial[c][i]
            if len(values) == 0:
                final_avg_epe[c, i] = 0.0
            else:
                # 去掉最大最小各 trim 比例的值
                final_avg_epe[c, i] = trim_mean(values, proportiontocut=trim)

    # 画图
    x_labels = [f"{int(i*100/n_bins)}~{int((i+1)*100/n_bins)}%" for i in range(n_bins)]
    x = range(n_bins)
    plt.figure(figsize=(6, 4))
    markers = ['o', 's', 'D', '^', 'v']  # 最多支持5个通道
    for c in range(C):
        plt.plot(x, final_avg_epe[c], marker=markers[c % len(markers)], label=flow_names[c])
    plt.xticks(x, x_labels)
    plt.xlabel("Velocity magnitude percentile")
    plt.ylabel("Average EPE (trimmed mean)")
    plt.legend(loc="upper right")
    plt.grid(True, alpha=0.7)
    plt.tight_layout()
    plt.show()


# ---------- 主程序修改 ----------

num_trials = 100
all_pred_flows = []
all_gt_flows = []

# 使用 tqdm 包装循环
for trial in tqdm(range(num_trials), desc="Running trials"):
    gt_list_trial = []
    flow_list_trial = []

    for i in range(3):
        flow_func = globals()[f"flow{i}"].evaluate
        cfg_evts = config_events()
        cfg_evts.time_length = 5
        cfg_evts.d = 0.5
        cfg_evts.density = 0.05

        event_loader = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func)
        evts, (u, v) = event_loader.load()
        flowgt = torch.stack([u, v], dim=0).cpu()
        gt_list_trial.append(flowgt)

        events = evts[(evts[:, 2] > 0) & (evts[:, 2] < 3)]

        cfg = GaussianFlowConfig(iterations=50, lr_j=0.001)
        res = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
        pred_h, pred_w = res.flow[0], res.flow[1]
        flow_pred = torch.tensor(np.stack([pred_h, pred_w], axis=0))
        flow_list_trial.append(flow_pred)

    flow_stack = torch.stack(flow_list_trial, dim=0)
    gt_stack = torch.stack(gt_list_trial, dim=0)

    all_pred_flows.append(flow_stack)
    all_gt_flows.append(gt_stack)

# 调用鲁棒分析函数
flow_names = ["Scene 0", "Scene 1", "Scene 2"]   # 请根据实际名称修改
analyze_error_robust(all_pred_flows, all_gt_flows, flow_names, n_bins=10, trim=0.1)

## Robustness investigation

- particle density

In [ ]:
density_results = {}

for i in tqdm(range(3), desc="methods"):
    flow_func = globals()[f"flow{i}"].evaluate
    cfg_evts = config_events()
    cfg_evts.time_length = 7
    cfg_evts.d = 0.5

    single_flow_results = {}
    for density in tqdm([i / 100 for i in range(1, 51, 5)], desc=f"{flow_names[i]}-Density", leave=False):
        single_density_rmse = []
        single_density_aee = []
        
        for _ in range(5):
            cfg_evts.density = density
    
            event_loader = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func)
            evts, (u, v) = event_loader.load()

            u_gt, v_gt = u.cpu().numpy(), v.cpu().numpy()

            events = evts[(evts[:, 2] > 0) & (evts[:, 2] < 5)]
            cfg = GaussianFlowConfig(lr_j=0.001)
            result = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
            
            pred_h, pred_w = result.flow[0], result.flow[1]
            res = PIVmetric(u_gt, v_gt, pred_h, pred_w, show=False)
            rmse, aee = res[0], res[1]
            single_density_rmse.append(rmse)
            single_density_aee.append(aee)

        rmse_mean = np.mean(np.sort(single_density_rmse)[1:-1])
        aee_mean = np.mean(np.sort(single_density_aee)[1:-1])

        single_flow_results[f"{density:.6f}"] = {
            "rmse": round(float(rmse_mean), 4),
            "aee": round(float(aee_mean), 4)}

    density_results[flow_names[i]] = single_flow_results

In [ ]:
x_vals, y_rmse_list, y_aee_list = extract_data(density_results)

save1_1_1 = 0
if save1_1_1 == 1:
    rmse_path = "./output/EXP_C/density-rmse.pdf"
    aee_path = "./output/EXP_C/density-aee.pdf"
else:
    rmse_path = None
    aee_path = None

use_Chinese = 0
if use_Chinese ==1:
    use_chinese = True
    size = [14, 14, 18]
else:
    use_chinese = False
    size = [10, 10, 12]

legend_names_en = flow_names
legend_names_zh = ["正弦流场", "旋转流场", "胞元流场"]
y_major_ticks = [0, 0.2, 0.4, 0.6]
x_major_ticks = list(np.arange(0.01, 0.11, 0.02))
x_minor_ticks = list(np.arange(0.0, 0.11, 0.02))

plot_curvegraph(x_data=x_vals, y_data_list=y_rmse_list, 
    legend_names_en=legend_names_en, legend_names_zh=legend_names_zh, 
    title_en=None, title_zh=None, 
    xlabel_en='Particle density $\\rho_{p}$ $[ppp]$', xlabel_zh='粒子密度 $\\rho_{p}$', 
    ylabel_en="RMSE $[px]$", ylabel_zh="RMSE $[px]$", 
    ticks_size=size[0], legend_size=size[1], label_size=size[2], 
    xlim=(0.008, 0.102), ylim=(0, 0.6), 
    x_major_ticks=x_major_ticks, x_minor_ticks=x_minor_ticks,
    y_major_ticks=y_major_ticks, y_minor_ticks=None,
    x_log=False, filename=rmse_path, use_chinese=use_chinese)

plot_curvegraph(x_data=x_vals, y_data_list=y_aee_list, 
    legend_names_en=legend_names_en, legend_names_zh=legend_names_zh, 
    title_en=None, title_zh=None, 
    xlabel_en='Particle density $\\rho_{p}$ $[ppp]$', xlabel_zh='粒子密度 $\\rho_{p}$', 
    ylabel_en="AEE $[px]$", ylabel_zh="AEE $[px]$", 
    ticks_size=size[0], legend_size=size[1], label_size=size[2], 
    xlim=(0.008, 0.102), ylim=(0, 0.6), 
    x_major_ticks=x_major_ticks, x_minor_ticks=x_minor_ticks,
    y_major_ticks=y_major_ticks, y_minor_ticks=None,
    x_log=False, filename=aee_path, use_chinese=use_chinese)

- noise ratio

In [ ]:
noise_results = {}

for i in tqdm(range(3), desc="flows"):
    flow_func = globals()[f"flow{i}"].evaluate
    cfg_evts = config_events()
    cfg_evts.time_length = 5
    cfg_evts.density = 0.05
    cfg_evts.d = 0.5
    
    event_loader = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func)
    evts, (u, v) = event_loader.load()
    u_gt, v_gt = u.cpu().numpy(), v.cpu().numpy()

    single_flow_results = {}

    for j, noise_ratio in enumerate(tqdm(np.arange(0, 0.21, 0.02), desc=f"  Noise ratio", leave=False)):
        rmse_list, aee_list = [], []

        for k in range(5):
            events = evts[(evts[:, 2] > 0) & (evts[:, 2] < 3)]
            noisy_events = add_event_noise(events, 256, 256 , noise_ratio = noise_ratio)
            
            cfg = GaussianFlowConfig()
            res = run_gaussian_flow(events=noisy_events, cfg=cfg, flowgt=None)
            pred_h, pred_w = res.flow[:2]
            rmse, aee = PIVmetric(u_gt, v_gt, pred_h, pred_w, show=False)[:2]

            rmse_list.append(float(rmse))
            aee_list.append(float(aee))

        rmse_mean = np.mean(np.sort(rmse_list)[1:-1])
        aee_mean = np.mean(np.sort(aee_list)[1:-1])

        single_flow_results[f"{noise_ratio:.2f}"] = {
            "rmse": round(rmse_mean, 4),
            "aee": round(aee_mean, 4)
        }

    noise_results[flow_names[i]] = single_flow_results

In [ ]:
x_vals, y_rmse_list, y_aee_list = extract_data(noise_results)

legend_names_en = flow_names
legend_names_zh = ["正弦流场", "旋转流场", "胞元流场"]
y_major_ticks = [0, 0.2, 0.4, 0.6]

save1_1_3 = 0
if save1_1_3 == 1:
    rmse_path = "./output/EXP_C/noise_ratio-rmse.pdf"
    aee_path = "./output/EXP_C/noise_ratio-aee.pdf"
else:
    rmse_path = None
    aee_path = None

use_Chinese = 0
if use_Chinese ==1:
    use_chinese = True
    size = [14, 14, 18]
else:
    use_chinese = False
    size = [10, 10, 12]
    
x_major_ticks = list(range(0, 21, 5))
x_minor_ticks = list(range(1, 21, 1))

plot_curvegraph(x_data=x_vals*100, y_data_list=y_rmse_list, 
    legend_names_en=legend_names_en, legend_names_zh=legend_names_zh, 
    title_en=None, title_zh=None, 
    xlabel_en="Noise ratio $R$ $[\%]$", xlabel_zh="噪声比例 $R$ $[\%]$", 
    ylabel_en="RMSE $[px]$", ylabel_zh="RMSE $[px]$", 
    ticks_size=size[0], legend_size=size[1], label_size=size[2], 
    xlim=(-0.5, 20.5), ylim=(0, 0.6), 
    x_major_ticks=x_major_ticks, x_minor_ticks=x_minor_ticks,
    y_major_ticks=y_major_ticks, y_minor_ticks=None,
    x_log=False, filename=rmse_path, 
    use_chinese=use_chinese)

plot_curvegraph(x_data=x_vals*100, y_data_list=y_aee_list, 
    legend_names_en=legend_names_en, legend_names_zh=legend_names_zh, 
    title_en=None, title_zh=None, 
    xlabel_en="Noise ratio $R$ $[\%]$", xlabel_zh="噪声比例 $R$ $[\%]$", 
    ylabel_en="AEE $[px]$", ylabel_zh="AEE $[px]$", 
    ticks_size=size[0], legend_size=size[1], label_size=size[2], 
    xlim=(-0.5, 20.5), ylim=(0, 0.6), 
    x_major_ticks=x_major_ticks, x_minor_ticks=x_minor_ticks,
    y_major_ticks=y_major_ticks, y_minor_ticks=None,
    x_log=False, filename=aee_path, 
    use_chinese=use_chinese)

- particle diameter

In [ ]:
diameter_results = {}

for i in tqdm(range(3), desc="Flow Type"):
    flow_func = globals()[f"flow{i}"].evaluate
    cfg_evts = config_events()
    cfg_evts.time_length = 5
    cfg_evts.density = 0.05

    single_flow_results = {}

    for diameter in tqdm(np.arange(0.1, 3.1, 0.3), desc=f"{flow_names[i]}-Diameter", leave=False):
        single_density_rmse = []
        single_density_aee = []

        for _ in range(5):
            cfg_evts.d = diameter

            event_loader = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func)
            evts, (u, v) = event_loader.load()

            u_gt, v_gt = u.cpu().numpy(), v.cpu().numpy()

            events = evts[(evts[:, 2] > 0) & (evts[:, 2] < 3)]

            res = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
            pred_h, pred_w = res.flow[:2]
            rmse, aee = PIVmetric(u_gt, v_gt, pred_h, pred_w, show=False)[:2]
            
            # 收集每次的rmse和aee
            single_density_rmse.append(rmse)
            single_density_aee.append(aee)


        # 去掉最大值和最小值后取平均
        if len(single_density_aee) > 2:
            rmse_mean = np.mean(np.sort(single_density_rmse)[1:-1])
            aee_mean = np.mean(np.sort(single_density_aee)[1:-1])
        else:
            rmse_mean = np.mean(single_density_rmse)
            aee_mean = np.mean(single_density_rmse)

        # 保存结果
        single_flow_results[f"{diameter:.1f}"] = {
            "rmse": round(float(rmse_mean), 4),
            "aee": round(float(aee_mean), 4)
        }

    # 将每个流场类型的结果存储到exp112_results中
    diameter_results[flow_names[i]] = single_flow_results

In [ ]:
x_vals, y_rmse_list, y_aee_list = extract_data(diameter_results)

save1_1_2 = 0
if save1_1_2 == 1:
    rmse_path = "./output/EXP_C/picture_for_paper/diameter-rmse.pdf"
    aee_path = "./output/EXP_C/picture_for_paper/diameter-aee.pdf"
else:
    rmse_path = None
    aee_path = None

use_Chinese = 0
if use_Chinese ==1:
    use_chinese = True
    size = [14, 14, 18]
else:
    use_chinese = False
    size = [10, 10, 12]

legend_names_en = ["sinusoidal case", "rotational case", "cellular case"]
legend_names_zh = ["正弦流场", "旋转流场", "胞元流场"]
y_major_ticks = [0, 0.05, 0.1, 0.15]
x_major_ticks = list(range(0, 6, 1))
x_minor_ticks = list(np.arange(0.2, 4.9, 0.2))

plot_curvegraph(x_data=x_vals, y_data_list=y_rmse_list, 
    legend_names_en=legend_names_en, legend_names_zh=legend_names_zh, 
    title_en=None, title_zh=None, 
    xlabel_en="Particle diameter $d_{p}$ $[px]$", xlabel_zh='粒子直径 $d_{p}$ $[px]$', 
    ylabel_en="RMSE $[px]$", ylabel_zh="RMSE $[px]$", 
    ticks_size=size[0], legend_size=size[1], label_size=size[2], 
    xlim=(0, 3.1), ylim=(0.1,0.30), 
    x_major_ticks=x_major_ticks, x_minor_ticks=x_minor_ticks,
    y_major_ticks=y_major_ticks, y_minor_ticks=None,
    x_log=False, filename=rmse_path, use_chinese=use_chinese)

plot_curvegraph(x_data=x_vals, y_data_list=y_aee_list, 
    legend_names_en=legend_names_en, legend_names_zh=legend_names_zh, 
    title_en=None, title_zh=None, 
    xlabel_en="Particle diameter $d_{p}$ $[px]$", xlabel_zh='粒子直径 $d_{p}$ $[px]$', 
    ylabel_en="AEE $[px]$", ylabel_zh="AEE $[px]$", 
    ticks_size=size[0], legend_size=size[1], label_size=size[2], 
    xlim=(0, 3.1), ylim=(0.05,0.3), 
    x_major_ticks=x_major_ticks, x_minor_ticks=x_minor_ticks,
    y_major_ticks=y_major_ticks, y_minor_ticks=None,
    x_log=False, filename=aee_path, use_chinese=use_chinese)